In [1]:
# %% [markdown]
# # Global Energy Transitions — Visualization Project
# **COMP 6934 W26 — Introduction to Data Visualization**
#
# This notebook merges two Our World in Data datasets (energy + CO₂) and produces
# four interactive Plotly visualizations exploring the global energy transition.
#
# **Datasets:**
# - `owid-energy-data.csv` — 23 232 rows × 130 columns (energy mix, electricity, consumption)
# - `owid-co2-data.csv` — 50 411 rows × 79 columns (CO₂ emissions, GHG, temperature change)
#
# **Visualizations:**
# 1. Animated Sankey — energy source flows for a selected country over time
# 2. Small-Multiple Stacked Area — electricity mix by world region
# 3. Animated Scatterplot — GDP per capita vs. renewable share, sized by CO₂
# 4. Interactive Parallel Coordinates — country energy profiles

---
## 1. Load and Inspect Data

In [2]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

In [3]:
# Load datasets
energy = pd.read_csv('owid-energy-data.csv')
co2 = pd.read_csv('owid-co2-data.csv')

print(f"Energy dataset: {energy.shape[0]:,} rows × {energy.shape[1]} columns")
print(f"CO2 dataset:    {co2.shape[0]:,} rows × {co2.shape[1]} columns")

Energy dataset: 23,232 rows × 130 columns
CO2 dataset:    50,411 rows × 79 columns


In [4]:
# Quick overview
print("=== Energy: year range ===")
print(f"  {energy.year.min()} – {energy.year.max()}")
print(
    f"  Countries (with ISO code): {energy[energy.iso_code.notna()].country.nunique()}")
print(
    f"  Aggregate regions: {energy[energy.iso_code.isna()].country.nunique()}")
print()
print("=== CO2: year range ===")
print(f"  {co2.year.min()} – {co2.year.max()}")
print(f"  Countries: {co2.country.nunique()}")

=== Energy: year range ===
  1900 – 2025
  Countries (with ISO code): 220
  Aggregate regions: 94

=== CO2: year range ===
  1750 – 2024
  Countries: 254


---
## 2. Merge and Clean

In [5]:
# Identify CO2-only columns (not already in energy)
shared_keys = ['country', 'year', 'iso_code', 'population', 'gdp']
co2_only_cols = [c for c in co2.columns if c not in energy.columns]
print(f"CO2-only columns to merge: {len(co2_only_cols)}")

# Merge: left join energy with co2-only columns on (country, year)
co2_subset = co2[['country', 'year'] + co2_only_cols].copy()
df = energy.merge(co2_subset, on=['country', 'year'], how='left')
print(f"Merged dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")

CO2-only columns to merge: 71
Merged dataset: 23,232 rows × 201 columns


In [6]:
# Separate countries from aggregates
countries_df = df[df.iso_code.notna()].copy()
print(f"Country-level rows: {countries_df.shape[0]:,}")

# OWID provides clean continent-level aggregates we can use directly
REGIONS = ['Africa', 'Asia', 'Europe',
           'North America', 'South America', 'Oceania']
regions_df = df[df.country.isin(REGIONS)].copy()
print(f"Region aggregate rows: {regions_df.shape[0]:,}")

# World aggregate
world_df = df[df.country == 'World'].copy()
print(f"World aggregate rows: {world_df.shape[0]:,}")

Country-level rows: 17,134
Region aggregate rows: 750
World aggregate rows: 125


In [7]:
# Define a continent mapping for individual countries
# We'll use the OWID aggregates for the regional stacked area chart,
# but need per-country continent labels for scatterplot and parallel coordinates.

# Use a simple mapping via iso_code prefix or manual assignment
# OWID doesn't include a continent column, so we build one from the region aggregates
# or use a lightweight approach.

# Approach: map countries to continents using a known mapping
CONTINENT_MAP = {
    'Africa': ['DZA', 'AGO', 'BEN', 'BWA', 'BFA', 'BDI', 'CPV', 'CMR', 'CAF', 'TCD', 'COM', 'COG', 'COD',
               'CIV', 'DJI', 'EGY', 'GNQ', 'ERI', 'SWZ', 'ETH', 'GAB', 'GMB', 'GHA', 'GIN', 'GNB', 'KEN',
               'LSO', 'LBR', 'LBY', 'MDG', 'MWI', 'MLI', 'MRT', 'MUS', 'MAR', 'MOZ', 'NAM', 'NER', 'NGA',
               'RWA', 'STP', 'SEN', 'SYC', 'SLE', 'SOM', 'ZAF', 'SSD', 'SDN', 'TZA', 'TGO', 'TUN', 'UGA',
               'ZMB', 'ZWE'],
    'Asia': ['AFG', 'ARM', 'AZE', 'BHR', 'BGD', 'BTN', 'BRN', 'KHM', 'CHN', 'CYP', 'GEO', 'IND', 'IDN',
             'IRN', 'IRQ', 'ISR', 'JPN', 'JOR', 'KAZ', 'KWT', 'KGZ', 'LAO', 'LBN', 'MYS', 'MDV', 'MNG',
             'MMR', 'NPL', 'PRK', 'OMN', 'PAK', 'PSE', 'PHL', 'QAT', 'SAU', 'SGP', 'KOR', 'LKA', 'SYR',
             'TWN', 'TJK', 'THA', 'TLS', 'TUR', 'TKM', 'ARE', 'UZB', 'VNM', 'YEM'],
    'Europe': ['ALB', 'AND', 'AUT', 'BLR', 'BEL', 'BIH', 'BGR', 'HRV', 'CZE', 'DNK', 'EST', 'FIN', 'FRA',
               'DEU', 'GRC', 'HUN', 'ISL', 'IRL', 'ITA', 'LVA', 'LTU', 'LUX', 'MLT', 'MDA', 'MNE', 'NLD',
               'MKD', 'NOR', 'POL', 'PRT', 'ROU', 'RUS', 'SRB', 'SVK', 'SVN', 'ESP', 'SWE', 'CHE', 'UKR',
               'GBR'],
    'North America': ['ATG', 'BHS', 'BRB', 'BLZ', 'CAN', 'CRI', 'CUB', 'DMA', 'DOM', 'SLV', 'GRD', 'GTM',
                      'HTI', 'HND', 'JAM', 'MEX', 'NIC', 'PAN', 'KNA', 'LCA', 'VCT', 'TTO', 'USA'],
    'South America': ['ARG', 'BOL', 'BRA', 'CHL', 'COL', 'ECU', 'GUY', 'PRY', 'PER', 'SUR', 'URY', 'VEN'],
    'Oceania': ['AUS', 'FJI', 'KIR', 'MHL', 'FSM', 'NRU', 'NZL', 'PLW', 'PNG', 'WSM', 'SLB', 'TON', 'TUV', 'VUT']
}

# Invert to iso_code -> continent
iso_to_continent = {}
for continent, codes in CONTINENT_MAP.items():
    for code in codes:
        iso_to_continent[code] = continent

countries_df = countries_df.copy()
countries_df['continent'] = countries_df['iso_code'].map(iso_to_continent)
print(
    f"Countries with continent assigned: {countries_df.continent.notna().sum():,} / {countries_df.shape[0]:,}")
print(
    f"Continent distribution:\n{countries_df.drop_duplicates('country').continent.value_counts()}")

Countries with continent assigned: 15,646 / 17,134
Continent distribution:
continent
Africa           54
Asia             49
Europe           39
North America    23
South America    12
Oceania          12
Name: count, dtype: int64


---
## 3. Visualization 1 — Animated Sankey: Energy Source Flows

Shows how a country's primary energy consumption is distributed across sources,
animated over time via a year slider. The user sees structural shifts in the
energy system — fossil flows shrinking (or not) as renewables grow.

In [8]:
def build_animated_sankey(country_name='World', year_range=(1990, 2023)):
    """
    Build an animated Sankey diagram showing energy source → category flows
    for a given country, with a slider over years.

    Sources: coal, oil, gas, nuclear, hydro, solar, wind, biofuel, other renewables
    Categories: Fossil Fuels, Low-Carbon (Nuclear), Renewables
    """
    # Energy source columns (TWh consumption)
    sources = {
        'Coal': 'coal_consumption',
        'Oil': 'oil_consumption',
        'Gas': 'gas_consumption',
        'Nuclear': 'nuclear_consumption',
        'Hydro': 'hydro_consumption',
        'Solar': 'solar_consumption',
        'Wind': 'wind_consumption',
        'Biofuel': 'biofuel_consumption',
        'Other Renewables': 'other_renewable_consumption'
    }

    # Category groupings
    categories = {
        'Fossil Fuels': ['Coal', 'Oil', 'Gas'],
        'Nuclear': ['Nuclear'],
        'Renewables': ['Hydro', 'Solar', 'Wind', 'Biofuel', 'Other Renewables']
    }

    subset = df[df.country == country_name].copy()
    subset = subset[(subset.year >= year_range[0]) &
                    (subset.year <= year_range[1])]
    subset = subset.sort_values('year')

    # Node labels: sources (0-8) + categories (9-11)
    source_names = list(sources.keys())
    category_names = list(categories.keys())
    all_labels = source_names + category_names

    # Color palettes
    source_colors = [
        '#4d4d4d',  # Coal - dark grey
        '#8c564b',  # Oil - brown
        '#1f77b4',  # Gas - blue
        '#ff7f0e',  # Nuclear - orange
        '#17becf',  # Hydro - cyan
        '#ffdd57',  # Solar - yellow
        '#2ca02c',  # Wind - green
        '#9467bd',  # Biofuel - purple
        '#7f7f7f',  # Other Renewables - grey
    ]
    category_colors = [
        '#d62728',  # Fossil Fuels - red
        '#ff7f0e',  # Nuclear - orange
        '#2ca02c',  # Renewables - green
    ]

    # Link colors (lighter versions of source colors)
    link_colors = [c.replace(')', ', 0.4)').replace('rgb', 'rgba') if 'rgb' in c
                   else f'rgba({int(c[1:3], 16)},{int(c[3:5], 16)},{int(c[5:7], 16)},0.4)'
                   for c in source_colors]

    # Build frames for each year
    frames = []
    slider_steps = []

    for year in subset.year.unique():
        row = subset[subset.year == year].iloc[0]

        link_sources = []
        link_targets = []
        link_values = []
        link_clrs = []

        for i, (src_name, col) in enumerate(sources.items()):
            val = row.get(col, 0)
            if pd.isna(val) or val <= 0:
                val = 0
            # Find which category this source belongs to
            for j, (cat_name, members) in enumerate(categories.items()):
                if src_name in members:
                    target_idx = len(source_names) + j
                    break
            if val > 0:
                link_sources.append(i)
                link_targets.append(target_idx)
                link_values.append(val)
                link_clrs.append(link_colors[i])

        frames.append(go.Frame(
            data=[go.Sankey(
                node=dict(
                    pad=20,
                    thickness=25,
                    line=dict(color='white', width=1),
                    label=all_labels,
                    color=source_colors + category_colors,
                ),
                link=dict(
                    source=link_sources,
                    target=link_targets,
                    value=link_values,
                    color=link_clrs,
                )
            )],
            name=str(year)
        ))

        slider_steps.append(dict(
            args=[[str(year)], dict(mode='immediate',
                                    frame=dict(duration=500, redraw=True),
                                    transition=dict(duration=300))],
            label=str(year),
            method='animate'
        ))

    # Initial frame
    fig = go.Figure(
        data=frames[0].data,
        frames=frames,
        layout=go.Layout(
            title=dict(
                text=f"Primary Energy Consumption by Source — {country_name}",
                font=dict(size=20)
            ),
            font=dict(size=13),
            height=600,
            width=1000,
            updatemenus=[dict(
                type='buttons',
                showactive=False,
                y=-0.05,
                x=0.0,
                xanchor='left',
                buttons=[
                    dict(label='▶ Play',
                         method='animate',
                         args=[None, dict(frame=dict(duration=700, redraw=True),
                                          fromcurrent=True,
                                          transition=dict(duration=400))]),
                    dict(label='⏸ Pause',
                         method='animate',
                         args=[[None], dict(frame=dict(duration=0, redraw=False),
                                            mode='immediate',
                                            transition=dict(duration=0))])
                ]
            )],
            sliders=[dict(
                active=0,
                steps=slider_steps,
                currentvalue=dict(prefix='Year: ', font=dict(size=16)),
                pad=dict(t=60),
                len=0.9,
                x=0.05,
            )],
            annotations=[dict(
                text="Energy flows from individual sources (left) to categories (right). Values in TWh.",
                showarrow=False, x=0.5, y=-0.12, xref='paper', yref='paper',
                font=dict(size=11, color='grey')
            )]
        )
    )

    return fig

In [9]:
fig1 = build_animated_sankey('World', (1990, 2023))
fig1.show()

In [11]:
# Also show for a specific country — China's dramatic shift
fig1b = build_animated_sankey('Canada', (1990, 2023))
fig1b.show()

---
## 4. Visualization 2 — Small-Multiple Stacked Area: Electricity Mix by Region

A faceted grid showing how the electricity generation mix (by source) has evolved
over time for each major world region. Enables cross-regional comparison of
transition speed and structure.

In [12]:
def build_electricity_mix_areas(year_range=(1990, 2023)):
    """
    Small-multiple stacked area charts of electricity generation by source
    for major world regions.
    """
    elec_sources = {
        'Coal': ('coal_electricity', '#4d4d4d'),
        'Gas': ('gas_electricity', '#1f77b4'),
        'Oil': ('oil_electricity', '#8c564b'),
        'Nuclear': ('nuclear_electricity', '#ff7f0e'),
        'Hydro': ('hydro_electricity', '#17becf'),
        'Wind': ('wind_electricity', '#2ca02c'),
        'Solar': ('solar_electricity', '#ffdd57'),
        'Biofuel': ('biofuel_electricity', '#9467bd'),
        'Other Renewables': ('other_renewable_electricity', '#7f7f7f'),
    }

    # ['Africa', 'Asia', 'Europe', 'North America', 'South America', 'Oceania']
    regions = REGIONS

    # Grid layout: 2 rows x 3 columns
    fig = make_subplots(
        rows=2, cols=3,
        subplot_titles=regions,
        shared_xaxes=True,
        shared_yaxes=False,
        vertical_spacing=0.1,
        horizontal_spacing=0.05,
    )

    for idx, region in enumerate(regions):
        row = idx // 3 + 1
        col = idx % 3 + 1

        rdata = regions_df[(regions_df.country == region) &
                           (regions_df.year >= year_range[0]) &
                           (regions_df.year <= year_range[1])].sort_values('year')

        if rdata.empty:
            continue

        show_legend = (idx == 0)  # Only show legend once

        for src_name, (col_name, color) in elec_sources.items():
            vals = rdata[col_name].fillna(0).values
            fig.add_trace(
                go.Scatter(
                    x=rdata.year,
                    y=vals,
                    mode='lines',
                    name=src_name,
                    stackgroup='one',
                    line=dict(width=0.5, color=color),
                    fillcolor=color,
                    showlegend=show_legend,
                    hovertemplate=f'{src_name}<br>%{{x}}<br>%{{y:,.0f}} TWh<extra>{region}</extra>',
                ),
                row=row, col=col
            )

    fig.update_layout(
        title=dict(
            text='Electricity Generation Mix by World Region (TWh)',
            font=dict(size=20),
        ),
        height=700,
        width=1200,
        legend=dict(
            orientation='h',
            yanchor='bottom',
            y=-0.12,
            xanchor='center',
            x=0.5,
            font=dict(size=11),
        ),
        hovermode='x unified',
        font=dict(size=11),
    )

    # Label y-axes
    for i in range(1, 3):
        fig.update_yaxes(title_text='TWh', row=i, col=1)

    return fig

In [13]:
fig2 = build_electricity_mix_areas((1990, 2023))
fig2.show()

---
## 5. Visualization 3 — Animated Scatterplot: Wealth vs. Decarbonization

A Gapminder-style animated bubble chart. Each bubble is a country.
- **X-axis:** GDP per capita (log scale)
- **Y-axis:** Renewable share of primary energy (%)
- **Bubble size:** Total CO₂ emissions
- **Color:** Continent
- **Animation:** Year (slider + play button)

This directly tests the hypothesis that wealthier nations decarbonize faster,
and reveals which countries outperform or underperform their income bracket.

In [14]:
def build_animated_scatterplot(year_range=(2000, 2023)):
    """
    Gapminder-style animated scatterplot:
    GDP per capita vs renewable share, sized by CO2, colored by continent.
    """
    scatter_df = countries_df[
        (countries_df.year >= year_range[0]) &
        (countries_df.year <= year_range[1]) &
        (countries_df.continent.notna()) &
        (countries_df.gdp.notna()) &
        (countries_df.population.notna()) &
        (countries_df.renewables_share_energy.notna()) &
        (countries_df.co2.notna())
    ].copy()

    scatter_df['gdp_per_capita'] = scatter_df['gdp'] / scatter_df['population']

    # Clamp very small values for log scale
    scatter_df = scatter_df[scatter_df.gdp_per_capita > 100]

    # Cap CO2 for size scaling (avoid single huge bubbles dominating)
    scatter_df['co2_display'] = scatter_df['co2'].clip(
        upper=scatter_df['co2'].quantile(0.98))

    # Continent colors
    continent_colors = {
        'Africa': '#e377c2',
        'Asia': '#ff7f0e',
        'Europe': '#1f77b4',
        'North America': '#2ca02c',
        'South America': '#d62728',
        'Oceania': '#17becf',
    }

    fig = px.scatter(
        scatter_df,
        x='gdp_per_capita',
        y='renewables_share_energy',
        size='co2_display',
        color='continent',
        color_discrete_map=continent_colors,
        hover_name='country',
        hover_data={
            'gdp_per_capita': ':,.0f',
            'renewables_share_energy': ':.1f',
            'co2': ':,.1f',
            'co2_display': False,
            'continent': False,
            'year': False,
        },
        animation_frame='year',
        animation_group='country',
        log_x=True,
        size_max=55,
        range_y=[-2, 100],
        labels={
            'gdp_per_capita': 'GDP per Capita (USD, log scale)',
            'renewables_share_energy': 'Renewable Share of Primary Energy (%)',
            'continent': 'Continent',
            'co2': 'CO₂ (Mt)',
        },
        title='Wealth vs. Decarbonization — Do Richer Countries Go Greener?',
    )

    fig.update_layout(
        height=650,
        width=1100,
        font=dict(size=12),
        legend=dict(
            title='Continent',
            font=dict(size=12),
        ),
        annotations=[dict(
            text="Bubble size = total CO₂ emissions (Mt). Data: Our World in Data.",
            showarrow=False, x=0.5, y=-0.1, xref='paper', yref='paper',
            font=dict(size=10, color='grey')
        )],
    )

    # Slow down animation
    fig.layout.updatemenus[0].buttons[0].args[1]['frame']['duration'] = 800
    fig.layout.updatemenus[0].buttons[0].args[1]['transition']['duration'] = 400

    return fig

In [15]:
fig3 = build_animated_scatterplot((2000, 2023))
fig3.show()

---
## 6. Visualization 4 — Interactive Parallel Coordinates: Country Energy Profiles

For a single recent year, each axis represents a different energy/emissions
attribute. Each line is a country, colored by continent. Brushing on any axis
filters the display, revealing clusters and outliers.

Dimensions:
- Renewable share of electricity (%)
- Fossil share of energy (%)
- Carbon intensity of electricity (gCO₂/kWh)
- Energy per capita (kWh)
- CO₂ per capita (tonnes)
- GDP per capita (USD)

In [16]:
def build_parallel_coordinates(target_year=2022):
    """
    Parallel coordinates plot of country energy profiles for a single year.
    Colored by continent (encoded as numeric for plotly parcoords).
    """
    pc_df = countries_df[
        (countries_df.year == target_year) &
        (countries_df.continent.notna())
    ].copy()

    # Select dimensions
    dims = {
        'renewables_share_elec': 'Renewable Share of Electricity (%)',
        'fossil_share_energy': 'Fossil Share of Energy (%)',
        'carbon_intensity_elec': 'Carbon Intensity (gCO₂/kWh)',
        'energy_per_capita': 'Energy per Capita (kWh)',
        'co2_per_capita': 'CO₂ per Capita (t)',
    }

    # Add GDP per capita
    pc_df['gdp_per_capita'] = pc_df['gdp'] / pc_df['population']
    dims['gdp_per_capita'] = 'GDP per Capita (USD)'

    # Drop rows with too many missing values
    pc_df = pc_df.dropna(subset=list(dims.keys()), thresh=4)

    # Encode continent as numeric for color
    continents = ['Africa', 'Asia', 'Europe',
                  'North America', 'South America', 'Oceania']
    continent_to_num = {c: i for i, c in enumerate(continents)}
    continent_colors_list = ['#e377c2', '#ff7f0e',
                             '#1f77b4', '#2ca02c', '#d62728', '#17becf']

    pc_df['continent_num'] = pc_df['continent'].map(continent_to_num)
    pc_df = pc_df[pc_df.continent_num.notna()]

    # Build dimensions list for parcoords
    dimension_list = []
    for col, label in dims.items():
        vals = pc_df[col].values
        valid = vals[~np.isnan(vals)]
        if len(valid) == 0:
            continue
        dimension_list.append(dict(
            range=[np.nanpercentile(vals, 1), np.nanpercentile(vals, 99)],
            label=label,
            values=vals,
        ))

    # Build colorscale from discrete continent colors
    n = len(continents)
    colorscale = []
    for i, color in enumerate(continent_colors_list):
        colorscale.append([i / (n - 1), color])

    fig = go.Figure(data=go.Parcoords(
        line=dict(
            color=pc_df['continent_num'],
            colorscale=colorscale,
            showscale=True,
            cmin=0,
            cmax=n - 1,
            colorbar=dict(
                title='Continent',
                tickvals=list(range(n)),
                ticktext=continents,
                len=0.8,
            )
        ),
        dimensions=dimension_list,
        labelangle=-30,
        labelside='top',
    ))

    fig.update_layout(
        title=dict(
            text=f'Country Energy Profiles — {target_year}',
            font=dict(size=20),
        ),
        height=600,
        width=1200,
        font=dict(size=11),
        margin=dict(l=80, r=80, t=100, b=40),
        annotations=[dict(
            text="Drag along any axis to filter. Each line is a country. Data: Our World in Data + CO₂ dataset.",
            showarrow=False, x=0.5, y=-0.06, xref='paper', yref='paper',
            font=dict(size=10, color='grey')
        )]
    )

    return fig

In [20]:
fig4 = build_parallel_coordinates(2020)
fig4.show()

---
## 7. Export All Figures as HTML

Each visualization is saved as a standalone interactive HTML file.

In [18]:
fig1.write_html('viz1_sankey_world.html', include_plotlyjs='cdn')
fig1b.write_html('viz1_sankey_china.html', include_plotlyjs='cdn')
fig2.write_html('viz2_electricity_mix_regions.html', include_plotlyjs='cdn')
fig3.write_html('viz3_animated_scatterplot.html', include_plotlyjs='cdn')
fig4.write_html('viz4_parallel_coordinates.html', include_plotlyjs='cdn')

print("All visualizations exported as HTML files.")
print("Files: viz1_sankey_world.html, viz1_sankey_china.html, viz2_electricity_mix_regions.html, viz3_animated_scatterplot.html, viz4_parallel_coordinates.html")

All visualizations exported as HTML files.
Files: viz1_sankey_world.html, viz1_sankey_china.html, viz2_electricity_mix_regions.html, viz3_animated_scatterplot.html, viz4_parallel_coordinates.html
